# Openflow Setup: S3 Parquet → Snowflake Iceberg Table

This notebook configures the Snowflake-side prerequisites for an Openflow deployment that ingests Parquet files from S3 into an Iceberg table.

**Workflow:**
1. Core Snowflake setup (OPENFLOW_ADMIN role + account privileges)
2. Create deployment (via Snowsight UI)
3. Runtime role + S3 network access
4. Create runtime (via Openflow UI)
5. Build NiFi flow on Openflow canvas (ListS3 → FetchS3Object → PutIcebergTable)

In [ ]:
%%sql -r core_setup
USE ROLE ACCOUNTADMIN;

CREATE ROLE IF NOT EXISTS OPENFLOW_ADMIN;

GRANT ROLE OPENFLOW_ADMIN TO USER XAVIZONE;

GRANT CREATE OPENFLOW DATA PLANE INTEGRATION ON ACCOUNT TO ROLE OPENFLOW_ADMIN;
GRANT CREATE OPENFLOW RUNTIME INTEGRATION ON ACCOUNT TO ROLE OPENFLOW_ADMIN;
GRANT CREATE COMPUTE POOL ON ACCOUNT TO ROLE OPENFLOW_ADMIN;

In [ ]:
%%sql -r user_config
USE ROLE ACCOUNTADMIN;

--ALTER USER XAVIZONE SET DEFAULT_ROLE = OPENFLOW_ADMIN;
ALTER USER XAVIZONE SET DEFAULT_ROLE = ACCOUNTADMIN;

ALTER USER XAVIZONE SET DEFAULT_SECONDARY_ROLES = ('ALL');

## Step 2: Create Deployment (UI)

1. In Snowsight, navigate to **Ingestion → Openflow**
2. Select **Launch Openflow**
3. Select **Create a deployment**
4. Choose **Snowflake** as the deployment location
5. Enter a deployment name (e.g. `s3_parquet_iceberg_deployment`)
6. Select **Create Deployment**

In [ ]:
%%sql -r runtime_role
USE ROLE ACCOUNTADMIN;

CREATE ROLE IF NOT EXISTS OPENFLOW_RUNTIME_ROLE_S3_PARQUET;
GRANT ROLE OPENFLOW_RUNTIME_ROLE_S3_PARQUET TO USER XAVIZONE;

GRANT USAGE, OPERATE ON WAREHOUSE COMPUTE_WH TO ROLE OPENFLOW_RUNTIME_ROLE_S3_PARQUET;

In [ ]:
%%sql -r db_setup
USE ROLE ACCOUNTADMIN;

CREATE DATABASE IF NOT EXISTS OPENFLOW_ICEBERG_DB;
CREATE SCHEMA IF NOT EXISTS OPENFLOW_ICEBERG_DB.RAW_SCHEMA;

GRANT USAGE ON DATABASE OPENFLOW_ICEBERG_DB TO ROLE OPENFLOW_RUNTIME_ROLE_S3_PARQUET;
GRANT USAGE ON SCHEMA OPENFLOW_ICEBERG_DB.RAW_SCHEMA TO ROLE OPENFLOW_RUNTIME_ROLE_S3_PARQUET;
GRANT CREATE TABLE, CREATE ICEBERG TABLE, CREATE STAGE ON SCHEMA OPENFLOW_ICEBERG_DB.RAW_SCHEMA
  TO ROLE OPENFLOW_RUNTIME_ROLE_S3_PARQUET;

In [ ]:
%%sql -r network_setup
USE ROLE ACCOUNTADMIN;

CREATE OR REPLACE NETWORK RULE OPENFLOW_ICEBERG_DB.RAW_SCHEMA.OPENFLOW_S3_NETWORK_RULE
  TYPE = HOST_PORT
  MODE = EGRESS
  VALUE_LIST = ('*.s3.us-west-2.amazonaws.com', '*.s3.amazonaws.com');

CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION OPENFLOW_S3_EAI
  ALLOWED_NETWORK_RULES = (OPENFLOW_ICEBERG_DB.RAW_SCHEMA.OPENFLOW_S3_NETWORK_RULE)
  ENABLED = TRUE
  COMMENT = 'EAI for Openflow S3 Parquet ingestion';

GRANT USAGE ON INTEGRATION OPENFLOW_S3_EAI TO ROLE OPENFLOW_RUNTIME_ROLE_S3_PARQUET;

## Step 4: Create Runtime (UI)

1. In the Openflow UI, go to the **Runtimes** tab
2. Select **Create a runtime**
3. Configure:
   - **Runtime Name**: e.g. `s3_parquet_runtime`
   - **Deployment**: select the deployment from Step 2
   - **Node Type**: choose based on data volume (start small)
   - **Min/Max Nodes**: `1 / 1` for dev, increase for production
   - **Snowflake Role**: `OPENFLOW_RUNTIME_ROLE_S3_PARQUET`
   - **External Access Integrations**: select `OPENFLOW_S3_EAI`
4. Select **Create**
5. Wait for runtime to provision (~2 minutes)

## Step 5: Build NiFi Flow on Openflow Canvas

Once the runtime is running, open it to access the NiFi canvas. Build this flow:

```
[ListS3] → [FetchS3Object] → [PutIcebergTable]
```

### Processor Configuration

**ListS3**
- Bucket: `<your-s3-bucket>`
- Region: `us-west-2` (adjust to your region)
- Prefix: `<path/to/parquet/files/>`
- AWS Credentials: configure `AWSCredentialsProviderControllerService` with IAM role or access keys

**FetchS3Object**
- Bucket: `${s3.bucket}` (from ListS3 attributes)
- Region: same as above
- AWS Credentials: same controller service as ListS3

**PutIcebergTable**
- Authentication Strategy: `SNOWFLAKE_MANAGED_TOKEN`
- Destination Database: `OPENFLOW_ICEBERG_DB`
- Destination Schema: `RAW_SCHEMA`
- Snowflake Role: `OPENFLOW_RUNTIME_ROLE_S3_PARQUET`
- Snowflake Warehouse: `COMPUTE_WH`
- Record Reader: configure a `ParquetReader` controller service

### Run the Flow
1. Enable all Controller Services (right-click canvas → Enable all Controller Services)
2. Right-click the process group → **Start**

In [ ]:
%%sql -r verify_volume
SELECT SYSTEM$VERIFY_EXTERNAL_VOLUME('iceberg_external_volume');